In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import train_test_split
import optuna
from utils.helpers import load_dataset, add_load_features, remove_nan_targets, remove_outliers

# Hyperparameter Search

This notebook was used to study how to run Bayesian hyperparameter search for an XGBoost model. At the end of this notebook is a table containing  optimal hyperparameters for each model type. 

# Load data

In [ ]:
project_root = Path.cwd().parent.parent.parent
interval = 60
multiplier = 4 if interval == 15 else 1
horizon = 24

path = project_root / f"data/time_series_{interval}.csv"
df = load_dataset(path)

# Extract only DE load

In [ ]:
TARGET = 'DE_load_actual_entsoe_transparency'
df = df[[TARGET]]
df = remove_outliers(df, TARGET)

# Feature engineering

In [ ]:
df = add_load_features(df, TARGET, interval, type="nowcast")
#df["target_forecast"] = df[TARGET].shift(-1 * horizon * multiplier)

# Remove NaN targets

In [ ]:
df = remove_nan_targets(df, TARGET)
#df = remove_nan_targets(df, "target_forecast")

# Convert data to numpy ndarrays

In [ ]:
X = df.drop(TARGET, axis=1).to_numpy()
y = df[TARGET].to_numpy()

"""X = df.drop([TARGET, "target_forecast"], axis=1).to_numpy()
y = df["target_forecast"].to_numpy()"""

print(f'X: {X.shape}, y: {y.shape}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [ ]:
def objective_nowcast(trial):
    
    # Define hyperparameter search space
    params = {
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0, step=0.05),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0, step=0.05),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5, step=0.5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10, step=0.5), # L2 regularization
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10, step=0.5),  # L1 regularization
    }
    
    model = xgb.XGBRegressor(
        **params,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        eval_metric='rmse',
        booster='gbtree',
        objective='reg:squarederror',
        n_estimators=3000,
        early_stopping_rounds=50,
    )
    
    # Time-series cross-validation
    tss = TimeSeriesSplit(n_splits=3, test_size=24*365*multiplier, gap=168*multiplier)
    
    scores = []
    for train_idx, val_idx in tss.split(X_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        y_pred = model.predict(X_val)
        rmse = np.sqrt(np.mean((y_val - y_pred) ** 2))
        scores.append(rmse)
    
    return np.mean(scores)  # Minimize mean RMSE

# Run optimization
study = optuna.create_study(
    direction='minimize',
    study_name='nowcast_optimization'
)

study.optimize(
    objective_nowcast,
    n_trials=50,
    n_jobs=4,
    show_progress_bar=True
)

# Get best parameters
best_params = study.best_params
print(f"Best RMSE: {study.best_value:.4f}")
print(f"Best params: {best_params}")

### Hyperparameter table with fixed n_estimator=10000 and early_stopping_rounds=200 

| Model                  |     Best RMSE | max_depth |        learning_rate | subsample | colsample_bytree | min_child_weight | gamma | reg_lambda | reg_alpha |
| ---------------------- | ------------: | --------: | -------------------: | --------: | ---------------: | ---------------: | ----: | ---------: | --------: |
| **NOWCAST 60min**      |  **549.0953** |         6 |  0.014247 |      0.75 |             0.95 |                6 |   4.5 |       10.0 |       5.0 |
| **NOWCAST 15min**      |  **392.2600** |        10 | 0.012147 |      0.55 |             0.95 |                7 |   4.0 |        3.5 |       2.0 |
| **FORECAST 60min +6**  | **1590.1732** |        10 | 0.012371 |      0.70 |             0.70 |                4 |   2.0 |        1.0 |       6.5 |
| **FORECAST 60min +12** | **1843.5065** |         9 | 0.026255 |      0.65 |             0.80 |                4 |   1.0 |        0.0 |       9.0 |
| **FORECAST 60min +24** | **2251.3173** |         9 | 0.017083 |      0.55 |             0.75 |                6 |   1.0 |        2.5 |       7.5 |


### Hyperparameter table with fixed n_estimator=3000 and early_stopping_rounds=50 

| Model                  |     Best RMSE | max_depth |        learning_rate | subsample | colsample_bytree | min_child_weight | gamma | reg_lambda | reg_alpha |
| ---------------------- | ------------: | --------: | -------------------: | --------: | ---------------: | ---------------: | ----: | ---------: | --------: |
| **NOWCAST 60min**      |       **552** |         6 | 0.030028 |      0.80 |             0.95 |                7 |   4.5 |        5.0 |       4.5 |
| **NOWCAST 15min**      |       **395** |         7 | 0.0582 |      0.85 |             1.0 |                5 |   2.5 |  1.5 |       3.0 |
| **FORECAST 60min +6**  |      **1614** |         8 | 0.030334 |      0.50 |             0.75 |                7 |   1.5 |        9.5 |       9.5 |
| **FORECAST 60min +12** | **1843.7881** |         9 | 0.034827 |      0.75 |             0.90 |                6 |   0.0 |        1.5 |       5.0 |
| **FORECAST 60min +24** |      **2247** |         7 | 0.052832 |      0.60 |             0.90 |                6 |   4.5 |        3.0 |       7.0 |
| **FORECAST 15min +6**  |      **1630** |        10 | 0.036856 |      0.75 |             0.65 |               10 |   5.0 |        3.0 |       2.0 |
| **FORECAST 15min +12** |      **1873** |         9 |  0.03427 |      0.60 |             0.90 |                8 |   2.5 |        6.5 |       3.5 |
| **FORECAST 15min +24** |      **2279** |         7 |  0.039773 |      0.95 |             0.90 |                8 |   3.0 |        5.5 |       3.5 |
